# 🚄 Notebook 4: Dual-Ended Differential Doppler Kinematics

Welcome to the **Differential Acoustic Doppler Laboratory**.

This notebook guides you through measuring the real-time velocity, acceleration, and position of an acoustic emitter moving along a 1D line using two opposing microphones (**Mic 1 at $x=0$** and **Mic 2 at $x=L$**).

---

### 🏛️ The Physical Power of the Dual-Ended Setup

1. **Buzzer Drift Cancellation:**
   Active buzzers drift with temperature and battery voltage. Because both microphones observe the same source simultaneously, their average tracks the drifting carrier in real time:
   $$f_0(t) = \frac{f_1(t) + f_2(t)}{2}$$

2. **Double Doppler Sensitivity ($2\times$):**
   Subtracting the two observed frequencies doubles the Doppler frequency split:
   $$\Delta f_{\text{diff}}(t) = f_2(t) - f_1(t) = 2 f_0(t) \cdot \frac{v(t)}{c(T)}$$

3. **Exact Drift-Immune Velocity Inversion:**
   $$v(t) = c(T) \cdot \left(\frac{f_2(t) - f_1(t)}{f_1(t) + f_2(t)}ight)$$

> **Experimental Modes:** This notebook works identically on an **air track** (with a glider) or on a **standard lab desk** (sliding/waving the buzzer by hand along a meter stick).

## 1. Hardware Initialization & Parameter Setup

Place your setup:
1. **Mic 1 (A0):** Left end ($x = 0$).
2. **Mic 2 (A1):** Right end ($x = L$, typically $1.0\,\text{m} - 2.0\,\text{m}$ apart).
3. **Buzzer:** Power the buzzer circuit with control wire plugged into $3.3\,\text{V}$ for steady $2610\,\text{Hz}$ tone.

In [ ]:
import json
from pathlib import Path
import numpy as np
from pynq_localizer import MicrophoneArrayOverlay, KinematicAnalytics

# 1. Initialize Hardware Overlay (50 kSPS per channel)
ol = MicrophoneArrayOverlay()

# 2. Physical Experiment Parameters
track_len_m = 1.50   # Physical distance between microphones in meters
temperature_c = 20.0 # Ambient temperature in °C
c_sound = KinematicAnalytics.speed_of_sound(temperature_c)

profile_path = Path("profiles/active_buzzer_2610hz.json")
if profile_path.exists():
    with open(profile_path, "r", encoding="utf-8") as f:
        p_data = json.load(f)
    f0 = p_data.get("f_res_hz", 2609.73)
else:
    f0 = 2609.73

print(f"✅ Hardware Overlay Active : {ol.fs_per_ch:.0f} SPS per channel")
print(f"✅ Speed of Sound c(T)     : {c_sound:.2f} m/s (at {temperature_c}°C)")
print(f"✅ Buzzer Nominal Carrier  : f0 = {f0:.2f} Hz")
print(f"✅ Doppler Scale Factor    : {2.0 * f0 / c_sound:.2f} Hz per (m/s)")

## 2. Quick 2-Second Motion Sanity Check

Verify that motion direction maps correctly:
* **Move Right (towards Mic 2):** Mic 2 is Blue-shifted, Mic 1 is Red-shifted $\implies v > 0$.
* **Move Left (towards Mic 1):** Mic 1 is Blue-shifted, Mic 2 is Red-shifted $\implies v < 0$.

In [ ]:
input("👉 Hold buzzer between mics. Press [Enter] and move it quickly RIGHT then LEFT...")

quick_run = ol.record_differential_flight(
    duration_sec=2.0,
    track_length_m=track_len_m,
    profile=profile_path,
    temperature_c=temperature_c
)

v_check = quick_run["velocity_cmps"]
max_right = np.max(v_check)
max_left = np.min(v_check)

print(f"   • Max Speed Right (+v) : {max_right:+5.1f} cm/s ({'✅ OK' if max_right > 5 else '⚠️ WEAK'})")
print(f"   • Max Speed Left  (-v) : {max_left:+5.1f} cm/s ({'✅ OK' if max_left < -5 else '⚠️ WEAK'})")
print(f"   • Peak Doppler Split   : {np.max(np.abs(quick_run['f_mic2_hz'] - quick_run['f_mic1_hz'])):.2f} Hz")

## 3. Continuous Multi-Second Flight Recording

Record a complete **5.0-second trajectory** as the glider coasts along the track (or as you slide the buzzer smoothly back and forth between the two microphones).

In [ ]:
duration_flight = 5.0

print("-" * 76)
print(f"📍 Launching {duration_flight:.1f}-second continuous flight recorder at 100 Hz trajectory rate...")
print("   1. Position buzzer/glider at the left side.")
print("   2. Press [Enter] and IMMEDIATELY PUSH it toward the right side.")
print("-" * 76)
input("👉 Press [Enter] and launch the motion...")

flight = ol.record_differential_flight(
    duration_sec=duration_flight,
    track_length_m=track_len_m,
    profile=profile_path,
    temperature_c=temperature_c,
    window_ms=40.0,
    hop_ms=10.0
)

t_sec = flight["times_sec"]
v_cmps = flight["velocity_cmps"]
pos_m = flight["position_m"]
f1 = flight["f_mic1_hz"]
f2 = flight["f_mic2_hz"]
f0_drift = flight["f0_common_hz"]
summary = flight["kinematics_summary"]

print(f"✅ Flight Captured: {len(t_sec)} time steps ({t_sec[0]:.2f}s to {t_sec[-1]:.2f}s)")

## 4. Interactive Kinematic Trajectory Visualization

Plot the three synchronized physical layers:
1. **Frequencies:** Shows the blue and red shifts crossing each other as direction changes, and the common-mode carrier $f_0(t)$.
2. **Velocity $v(t)$:** Shows forward acceleration, coasting deceleration, and direction reversals.
3. **Position $x(t)$:** Reconstructs the 1D position of the source between the two microphones.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.08,
    subplot_titles=(
        "<b>1. Dual Observed Frequencies (f1 Blue vs f2 Red) & Common Drift f0(t) [Hz]</b>",
        f"<b>2. Instantaneous Differential Velocity v(t) [cm/s] (Max = {np.max(np.abs(v_cmps)):.1f} cm/s)</b>",
        "<b>3. Integrated 1D Position Along Track x(t) [m]</b>"
    )
)

# Panel 1: Frequency Curves
fig.add_scatter(x=t_sec, y=f1, mode="lines", line=dict(color="#00FFCC", width=1.8), name="Mic 1 (Left)", row=1, col=1)
fig.add_scatter(x=t_sec, y=f2, mode="lines", line=dict(color="#FF007F", width=1.8), name="Mic 2 (Right)", row=1, col=1)
fig.add_scatter(x=t_sec, y=f0_drift, mode="lines", line=dict(color="#FFA500", width=1.5, dash="dash"), name="Carrier f0(t)", row=1, col=1)

# Panel 2: Velocity Curve
fig.add_scatter(x=t_sec, y=v_cmps, mode="lines+markers", marker=dict(size=3), line=dict(color="#FFD600", width=2.0), name="v_diff (cm/s)", row=2, col=1)
fig.add_hline(y=0.0, line=dict(color="gray", dash="dash"), annotation_text="Stationary (v = 0)", row=2, col=1)

# Panel 3: Position Trajectory
fig.add_scatter(x=t_sec, y=pos_m, mode="lines", line=dict(color="#76FF03", width=2.2), name="Position x(t)", row=3, col=1)
fig.add_hline(y=0.0, line=dict(color="white", dash="dot"), annotation_text="Left End (0 m)", row=3, col=1)
fig.add_hline(y=track_len_m, line=dict(color="white", dash="dot"), annotation_text=f"Right End ({track_len_m} m)", row=3, col=1)

fig.update_layout(template="plotly_dark", height=750, margin=dict(l=55, r=25, t=40, b=30))
fig.update_yaxes(title="Frequency (Hz)", row=1, col=1)
fig.update_yaxes(title="Velocity (cm/s)", row=2, col=1)
fig.update_yaxes(title="Position (m)", range=[-0.05, track_len_m + 0.05], row=3, col=1)
fig.update_xaxes(title="Time (Seconds)", row=3, col=1)
fig.show()

## 5. Aerodynamic Damping & Collision Analysis

The tracker automatically extracts physical parameters from the velocity trajectory:
* **Viscous Drag Coefficient ($\gamma$):** Fits linear damping $v(t) = v_0 e^{-\gamma t}$ on undisturbed coasting segments.
* **Bumper Restitution ($e$):** Measures kinetic energy retention during direction reversals ($e = |v_{\text{after}}| / |v_{\text{before}}|$).

In [ ]:
print("=" * 76)
print("📊 AERODYNAMIC & COLLISION KINEMATICS REPORT")
print("=" * 76)
print(f"  • Max Forward Velocity (+v) : {summary['max_forward_velocity_mps']*100:+5.1f} cm/s ({summary['max_forward_velocity_mps']:+.3f} m/s)")
print(f"  • Max Reverse Velocity (-v) : {summary['max_reverse_velocity_mps']*100:+5.1f} cm/s ({summary['max_reverse_velocity_mps']:+.3f} m/s)")
print(f"  • Collisions / Reversals    : {summary['total_collisions_detected']}")
if np.isfinite(summary.get("mean_coefficient_of_restitution", np.nan)):
    print(f"  • Coefficient of Restitution: e = {summary['mean_coefficient_of_restitution']:.3f} (Collision Elasticity)")
print(f"  • Viscous Drag Coefficient  : γ = {summary['viscous_drag_gamma']:.4f} s⁻¹")
print(f"  • Net Distance Traveled     : {pos_m[-1] - pos_m[0]:.3f} m")
print("=" * 76)

## 6. Real-Time 100 Hz Live Streaming Dashboard

Launch the interactive dashboard. Click **Tab 3 (🔀 Dual Overlay & Doppler)** and observe **Row 4** streaming real-time differential velocity $v_{\text{diff}}(t)$ live at $30\,\text{FPS}$ as you move the buzzer!

In [ ]:
# Launch the real-time 4-tab live dashboard with Doppler enabled
app = ol.kinematics_dashboard(
    window_duration_sec=10.0,
    hop_ms=10.0,
    aoa_mic_distance_m=0.05
)

## 7. Clean Hardware Teardown

When finished, cleanly stop the dashboard background threads and release FPGA DMA memory buffers.

In [ ]:
if 'app' in locals():
    app.stop()
ol.close()
print("🔒 FPGA hardware and DMA buffers cleanly released.")